# 🎬 Text-to-Video Part B: Training, FVD & System Design

## Training Strategies

### Joint Image-Video Training vs. Pretrain-then-Finetune

**Why image data helps:** Video datasets are scarce and expensive to label. Images are abundant and teach the model strong spatial priors (object appearance, texture, lighting). Since a video frame *is* an image, a model that understands images already knows what a single frame should look like — the video-specific job is learning *motion and temporal coherence*.

| Strategy | Description | Pros | Cons |
|---|---|---|---|
| **Joint image-video training** | Mix image and video batches; treat an image as a 1-frame video | Efficiently uses image data throughout training; prevents forgetting | Requires careful loss balancing; images carry no temporal signal |
| **Pretrain-then-finetune** | Train a strong image/text model first, then finetune on video | Clean separation of concerns; easy to reuse existing image checkpoints | Two-stage cost; spatial knowledge may shift during video finetune |
| **Video-only training** | Train exclusively on video from scratch | No domain-mismatch issues | Data-hungry; typically produces lower visual quality |

**Practical note:** Sora, VideoLDM, and Make-A-Video all leverage image pretraining. The common trick is to freeze spatial weights and add temporal attention layers that are fine-tuned on video.

In [ ]:
import torch

# Video tensor: (Batch, Time, Height, Width, Channels)
B, T, H, W, C = 2, 16, 32, 32, 3
video = torch.randn(B, T, H, W, C)
print(f"Input video:          {tuple(video.shape)}")

# Patch sizes
pt, ph, pw = 4, 8, 8  # temporal, spatial patches

# 1. Reshape to expose patch grid dimensions
v = video.view(B, T//pt, pt, H//ph, ph, W//pw, pw, C)
print(f"After patch split:    {tuple(v.shape)}")

# 2. Permute so patch grid dims are contiguous, then flatten patches
v = v.permute(0, 1, 3, 5, 2, 4, 6, 7).contiguous()
print(f"After permute:        {tuple(v.shape)}")

n_patches = (T//pt) * (H//ph) * (W//pw)
patch_dim  = pt * ph * pw * C
v = v.view(B, n_patches, patch_dim)
print(f"Flattened patches:    {tuple(v.shape)}")
print(f"  -> {n_patches} tokens of dim {patch_dim} each")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simulate a 12-FPS clip (T=6 frames) as random grayscale frames
np.random.seed(0)
T_low = 6
low_fps_frames = [np.clip(np.random.randn(32, 32) * 0.3 + 0.5, 0, 1) for _ in range(T_low)]

# Linear interpolation: insert one midpoint frame between every pair -> 12 FPS -> 24 FPS
high_fps_frames = []
for i in range(len(low_fps_frames) - 1):
    high_fps_frames.append(low_fps_frames[i])
    mid = 0.5 * low_fps_frames[i] + 0.5 * low_fps_frames[i + 1]  # linear interp
    high_fps_frames.append(mid)
high_fps_frames.append(low_fps_frames[-1])

# Plot first 6 frames of each
fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for i in range(6):
    axes[0, i].imshow(low_fps_frames[i], cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f"12fps f{i}", fontsize=8); axes[0, i].axis('off')
    axes[1, i].imshow(high_fps_frames[i], cmap='gray', vmin=0, vmax=1)
    axes[1, i].set_title(f"24fps f{i}", fontsize=8); axes[1, i].axis('off')
axes[0, 0].set_ylabel("Original", fontsize=9)
axes[1, 0].set_ylabel("Upsampled", fontsize=9)
plt.suptitle("Temporal Super-Resolution: 12 FPS → 24 FPS (linear interp)", fontsize=11)
plt.tight_layout(); plt.show()
print(f"Frames: {T_low} → {len(high_fps_frames)}")

## Evaluation Metrics

### Frechet Video Distance (FVD)
FVD extends FID from images to videos. Instead of InceptionV3, it uses an **I3D** (Inflated 3D ConvNet) network pretrained on Kinetics to extract spatiotemporal features. The Frechet distance between feature distributions of real and generated clips is computed — lower is better.

$$\text{FVD} = \|\mu_r - \mu_g\|^2 + \text{Tr}\left(\Sigma_r + \Sigma_g - 2(\Sigma_r \Sigma_g)^{1/2}\right)$$

### Summary of Metrics

| Metric | What it measures | Tool | Note |
|---|---|---|---|
| **FVD** | Realism + diversity of video clips | I3D features + Frechet dist | Primary video quality metric |
| **Per-frame FID** | Static image quality of individual frames | InceptionV3 features | Ignores temporal structure |
| **CLIPScore** | Semantic alignment of video to text prompt | CLIP (video frames vs. text) | Higher = better alignment |
| **Temporal Consistency** | Smoothness across frames | CLIP/DINO cosine sim between adjacent frames | Detects flickering / incoherence |
| **IS (Inception Score)** | Sharpness + diversity (legacy) | InceptionV3 | Less used for video now |

In [ ]:
import numpy as np
from scipy.linalg import sqrtm

def frechet_distance(mu1, sigma1, mu2, sigma2):
    """Compute Frechet distance between two multivariate Gaussians."""
    diff = mu1 - mu2
    # Numerically stable matrix square root
    cov_mean = sqrtm(sigma1 @ sigma2)
    if np.iscomplexobj(cov_mean):
        cov_mean = cov_mean.real
    return float(diff @ diff + np.trace(sigma1 + sigma2 - 2 * cov_mean))

np.random.seed(42)
feature_dim = 64   # In real FVD this is 400 (I3D output)
n_real, n_gen = 100, 100

# Simulate I3D-like features for real and generated video batches
real_feats = np.random.randn(n_real, feature_dim) * 1.0
good_feats = np.random.randn(n_gen, feature_dim) * 1.0 + 0.1  # close distribution
bad_feats  = np.random.randn(n_gen, feature_dim) * 2.0 + 2.0  # distant distribution

def stats(feats):
    return feats.mean(axis=0), np.cov(feats, rowvar=False)

mu_r, s_r = stats(real_feats)
fvd_good = frechet_distance(mu_r, s_r, *stats(good_feats))
fvd_bad  = frechet_distance(mu_r, s_r, *stats(bad_feats))

print(f"Simplified FVD (good generator): {fvd_good:.2f}  <- low = realistic")
print(f"Simplified FVD (bad generator):  {fvd_bad:.2f}  <- high = unrealistic")

## Overall System Design

### Cascade Pipeline

```
Text Prompt
    │
    ▼
┌─────────────────────────────┐
│  Text Encoder (T5 / CLIP)   │  → text embeddings
└─────────────┬───────────────┘
              │
              ▼
┌─────────────────────────────┐
│  Latent Video Generator     │  Diffusion / AR in compressed latent space
│  (DiT or U-Net backbone)    │  Low res, low FPS  e.g. 16x16, 4fps
│  + Temporal Attention       │
└─────────────┬───────────────┘
              │  latent video z
              ▼
┌─────────────────────────────┐
│  VAE Decoder (Visual Dec.)  │  Decode z → pixel frames
│  Spatial: 16x16 → 256x256  │
└─────────────┬───────────────┘
              │  low-res pixel video
              ▼
┌─────────────────────────────┐
│  Spatial Super-Resolution   │  e.g. 256x256 → 1024x1024
│  (Video SR diffusion model) │
└─────────────┬───────────────┘
              │  hi-res, low-FPS video
              ▼
┌─────────────────────────────┐
│  Temporal Super-Resolution  │  e.g. 4fps → 24fps
│  (Frame interpolation model)│
└─────────────┬───────────────┘
              │
              ▼
     Final Video Output
  1024x1024 @ 24fps
```

### Interview Cheat Sheet

| Concept | One-liner |
|---|---|
| **LDM (Latent Diffusion Model)** | Run diffusion in VAE latent space, not pixel space — much cheaper |
| **Temporal attention** | Self-attention along the time axis between frames at same spatial position |
| **FVD** | Frechet distance on I3D spatiotemporal features; main video quality metric |
| **DiT vs U-Net** | DiT = pure transformer on patches; U-Net = CNN with skip connections. DiT scales better. |
| **3D patchification** | Split (T,H,W) into cubes → sequence of tokens for transformer input |
| **Classifier-free guidance** | Train with and without text; at inference push away from unconditional prediction |

## Quiz

**Q1. Why do text-to-video models almost always use a cascaded architecture instead of generating full-resolution video directly?**

<details>
<summary>Show answer</summary>

Generating high-resolution, high-FPS video in a single pass requires modeling an enormous number of pixels/tokens simultaneously — computationally intractable. Cascades break this into manageable stages (latent generation → spatial upsampling → temporal upsampling), each operating on a smaller representation.

</details>

---

**Q2. FVD uses an I3D network rather than InceptionV3. What does I3D capture that InceptionV3 cannot?**

<details>
<summary>Show answer</summary>

I3D uses inflated 3D convolutions that operate over both space and time simultaneously, capturing *motion patterns and temporal dynamics*. InceptionV3 has no temporal dimension — it treats each frame independently, so it cannot penalise videos with inconsistent or implausible motion.

</details>

---

**Q3. When fine-tuning an image diffusion model for video, which weights are typically frozen and which are added/fine-tuned?**

<details>
<summary>Show answer</summary>

Spatial (2D) attention and convolution weights are typically **frozen** to preserve image quality. New **temporal attention layers** (1D self-attention along the time axis) are inserted and trained from scratch on video data. This allows the model to learn motion while retaining high-quality spatial features.

</details>